# Week 07 — Content Action Playbook
**Lane:** CTR / Engagement Opportunity Scoring (Lane 4)
**Builds on:** `work/notebooks/w04_baseline_score.ipynb` (the rule), `w05_model.ipynb` (the
model), `w06_validation_audit.ipynb` (the honest audit)
**Seed:** 42

This turns the validated Week 4-6 output into something a human team could actually use: ranked
actions with reason codes, an archetype-level view, the decay insight, intended use, limits,
human-review rules, cost/value thinking, and monitoring triggers. **This is a decision-support
plan, not a production system** -- nothing here ships automatically.

Sections: 1) Ranked actions + reason codes, 2) Intended use and limits, 3) Human review + the
no-go list, 4) Monitoring / retrain triggers, 5) Exports for the paper, 6) Self-check.

All claims use public-safe language -- observed / measured / directional / decision-support.

In [1]:
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Rida-create/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    cwd = Path.cwd()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "data" / "raw" / "content_refresh_anonymized.csv").exists():
            os.chdir(candidate)
            break

import json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

sys.path.insert(0, "scripts")
from ml_utils import NUMERIC_COLUMNS, CATEGORICAL_COLUMNS

pd.set_option("display.width", 140)
SEED = 42
np.random.seed(SEED)

OUT_DIR = Path("work/outputs")
FIG_DIR = Path("work/figures")
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)
print("Setup OK.")

Setup OK.


In [2]:
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

for c in NUMERIC_COLUMNS:
    df[c] = pd.to_numeric(df[c], errors="coerce")
for c in CATEGORICAL_COLUMNS:
    df[c] = df[c].fillna("unknown").astype(str).replace({"": "unknown", "nan": "unknown"})
for c in NUMERIC_COLUMNS:
    df[c] = df[c].replace([np.inf, -np.inf], np.nan).fillna(0)

df = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].copy()
df = df.drop_duplicates(subset=["content_id"]).reset_index(drop=True)
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

print(f"Rows: {len(df):,}  |  clients: {df['client_id'].nunique()}")

Rows: 30,000  |  clients: 32


## 1. Ranked actions + reason codes

Regenerating the Week-4 rule -- same formula, same volume floor, same reason-code priority order
-- rather than depending on a committed CSV that may not exist on this runtime (see `w05`/`w06`
for why: Colab gives each notebook its own separate runtime by default).

In [3]:
VOLUME_FLOOR = 300
scored = df.copy()
scored["eligible"] = (scored["avg_position"] > 0) & (scored["impressions_90d"] >= VOLUME_FLOOR)

tier_expected_ctr = scored.loc[scored["eligible"]].groupby("position_tier")["ctr"].median()
scored["expected_ctr_tier"] = scored["position_tier"].map(tier_expected_ctr)
scored["ctr_gap"] = scored["ctr"] - scored["expected_ctr_tier"]

def pct_rank(s):
    return s.rank(pct=True, method="average")

scored["ctr_gap_score"] = 0.0
scored["volume_score"] = 0.0
scored["engagement_gap_score"] = 0.0
elig_idx = scored.index[scored["eligible"]]
scored.loc[elig_idx, "ctr_gap_score"] = pct_rank(-scored.loc[elig_idx, "ctr_gap"])
scored.loc[elig_idx, "volume_score"] = pct_rank(np.log1p(scored.loc[elig_idx, "impressions_90d"]))

engagement_mask = (
    scored["eligible"] & (scored["sessions_90d"] >= 30)
    & (scored["engagement_rate"] > 0) & (scored["engagement_rate"] < 30)
)
eng_gap_raw = (30 - scored.loc[engagement_mask, "engagement_rate"]).clip(lower=0)
scored.loc[engagement_mask, "engagement_gap_score"] = pct_rank(eng_gap_raw)

scored["opportunity_score"] = (
    0.55 * scored["ctr_gap_score"] + 0.25 * scored["volume_score"] + 0.20 * scored["engagement_gap_score"]
).clip(0, 1)
scored.loc[~scored["eligible"], "opportunity_score"] = 0.0

def reason_and_action(row):
    if not row["eligible"]:
        return "insufficient_volume_or_position", "monitor"
    if row["ctr_gap"] < 0 and row["ctr_gap_score"] >= row["engagement_gap_score"]:
        return "ctr_fix_candidate", "rewrite_title_meta"
    if row["engagement_gap_score"] > 0:
        return "engagement_review_candidate", "improve_onpage_engagement"
    return "on_track", "monitor"

reason_action = scored.apply(reason_and_action, axis=1, result_type="expand")
scored["reason_code"] = reason_action[0]
scored["action_label"] = reason_action[1]
scored["rank"] = scored["opportunity_score"].rank(method="first", ascending=False).astype(int)

print("reason_code counts:")
print(scored["reason_code"].value_counts())

reason_code counts:
reason_code
insufficient_volume_or_position    11248
ctr_fix_candidate                   8338
on_track                            6239
engagement_review_candidate         4175
Name: count, dtype: int64


**Archetype -> action mapping.** Reason codes describe *why* a row scored the way it did;
archetypes group rows into the handful of recognizable situations a reviewer will actually see
repeatedly, each with a default action. This adds `position_tier` context on top of `reason_code`
because Week 4's own audit found the rule's absolute-gap math structurally favors `page_1` --
splitting `ctr_fix_candidate` by tier keeps that bias visible instead of hiding it behind one
generic label.

In [4]:
def archetype(row):
    if row["reason_code"] == "insufficient_volume_or_position":
        return "Not Enough Data Yet"
    if row["reason_code"] == "on_track":
        return "On Track"
    if row["reason_code"] == "engagement_review_candidate":
        return "Ranking Well, Low Engagement"
    if row["reason_code"] == "ctr_fix_candidate":
        if row["position_tier"] in ("top_3", "page_1", "striking"):
            return "Visible & Underclicked"
        return "Buried & Underperforming"
    return "Other"

scored["archetype"] = scored.apply(archetype, axis=1)

archetype_table = (
    scored.groupby(["archetype", "reason_code", "action_label"])
    .size()
    .rename("n_rows")
    .reset_index()
    .sort_values("n_rows", ascending=False)
)
archetype_table

,archetype,reason_code,action_label,n_rows
1,Not Enough Data Yet,insufficient_volume_or_position,monitor,11248
2,On Track,on_track,monitor,6239
4,Visible & Underclicked,ctr_fix_candidate,rewrite_title_meta,6140
3,"Ranking Well, Low Engagement",engagement_review_candidate,improve_onpage_engagement,4175
0,Buried & Underperforming,ctr_fix_candidate,rewrite_title_meta,2198


In [5]:
display_cols = ["rank", "content_id", "archetype", "action_label", "reason_code",
                "opportunity_score", "ctr", "expected_ctr_tier", "position_tier", "impressions_90d"]
scored.sort_values("rank")[display_cols].head(15)

,rank,content_id,archetype,action_label,reason_code,opportunity_score,ctr,expected_ctr_tier,position_tier,impressions_90d
9193,1,content_c1fe78bc4e37,Visible & Underclicked,rewrite_title_meta,ctr_fix_candidate,0.945516,0.03,0.23,page_1,134055
21782,2,content_2532dc9f4e3f,"Ranking Well, Low Engagement",improve_onpage_engagement,engagement_review_candidate,0.939491,0.06,0.23,page_1,51540
8399,3,content_b49efa4db88a,Visible & Underclicked,rewrite_title_meta,ctr_fix_candidate,0.936273,0.03,0.23,page_1,46866
10646,4,content_d0cc5baa4995,Visible & Underclicked,rewrite_title_meta,ctr_fix_candidate,0.932062,0.03,0.23,page_1,83651
10040,5,content_4d7e5bd31c6c,Visible & Underclicked,rewrite_title_meta,ctr_fix_candidate,0.931695,0.04,0.23,page_1,26707
21290,6,content_65668a45a72d,Visible & Underclicked,rewrite_title_meta,ctr_fix_candidate,0.931104,0.04,0.23,page_1,22088
29775,7,content_896bf2cc27b7,Visible & Underclicked,rewrite_title_meta,ctr_fix_candidate,0.921889,0.04,0.23,page_1,66359
3394,8,content_36ff89c8214e,Visible & Underclicked,rewrite_title_meta,ctr_fix_candidate,0.917741,0.05,0.23,page_1,295097
24868,9,content_7787faf11636,"Ranking Well, Low Engagement",improve_onpage_engagement,engagement_review_candidate,0.909919,0.09,0.23,page_1,90393
13798,10,content_a089e21690a5,Visible & Underclicked,rewrite_title_meta,ctr_fix_candidate,0.909829,0.05,0.23,page_1,12727


## 2. Intended use and limits

**The decay/refresh insight.** Week 5/6 found `content_age_days` was one of the stronger
predictors of the decline label, in the negative direction -- worth checking directly with a
plain bucket table rather than just citing the earlier correlation number.

In [6]:
age_order = ["31-90", "91-180", "181-365", "365+"]
decay_table = df.groupby("age_tier")["is_declining_label"].agg(n="count", decline_rate="mean").reindex(age_order)
decay_table["decline_rate"] = (decay_table["decline_rate"] * 100).round(1)
decay_table

,n,decline_rate
age_tier,,
31-90,492,66.9
91-180,11780,62.6
181-365,11368,51.5
365+,6360,42.6


## 3. Human review + the no-go list

**Human-review rules.** Every row that reaches a person comes with its archetype, reason code,
and the underlying numbers (`ctr` vs `expected_ctr_tier`, `impressions_90d`, `position_tier`) --
never just a bare score. Specifically:
- **`Visible & Underclicked`** -- a person should glance at the actual SERP before rewriting
  anything: heavy ad density, a shopping carousel, or a featured snippet above the page can cap
  CTR regardless of title quality (Week 5's error analysis found exactly this pattern).
- **`Buried & Underperforming`** -- needs a second check on whether the real issue is intent
  mismatch rather than a fixable metadata problem, per `work/ml_task_framing.md`'s own caution
  ("low CTR isn't automatically a title/meta problem").
- **`Ranking Well, Low Engagement`** -- needs a human to actually read the page. A short,
  fast-answer page can have low engagement *because it's working*, not failing.
- **`Not Enough Data Yet` / `On Track`** -- no action, but also no "all clear" -- these mean *no
  evidence of a problem*, not *confirmed healthy*.

**Cost/value thinking.** Effort levels below are a stated assumption for a typical team, not a
measured figure -- swap in real numbers before using this for planning. What *is* measured is
where the volume actually sits, which is what should drive triage order within an archetype.

In [7]:
cost_value = pd.DataFrame([
    {"archetype": "Visible & Underclicked", "assumed_effort": "~15-30 min (rewrite title/meta)",
     "worth_it_when": "impressions_90d is high enough that a CTR gap translates to real clicks"},
    {"archetype": "Buried & Underperforming", "assumed_effort": "~15-30 min, but often needs a second look first",
     "worth_it_when": "only after confirming it's not an intent mismatch (see human-review rules)"},
    {"archetype": "Ranking Well, Low Engagement", "assumed_effort": "hours (needs actual content/UX review)",
     "worth_it_when": "sessions_90d is high enough that the engagement pattern is not just noise"},
    {"archetype": "On Track / Not Enough Data Yet", "assumed_effort": "~0 (monitor only)",
     "worth_it_when": "never actioned directly -- re-evaluated next cycle"},
])
print(cost_value.to_string(index=False))
print()
print("Impressions_90d distribution within 'Visible & Underclicked' (where triage order should focus):")
print(scored.loc[scored["archetype"] == "Visible & Underclicked", "impressions_90d"].describe()[["min","25%","50%","75%","max"]].round(0))

                     archetype                                  assumed_effort                                                              worth_it_when
        Visible & Underclicked                 ~15-30 min (rewrite title/meta)    impressions_90d is high enough that a CTR gap translates to real clicks
      Buried & Underperforming ~15-30 min, but often needs a second look first only after confirming it's not an intent mismatch (see human-review rules)
  Ranking Well, Low Engagement          hours (needs actual content/UX review)  sessions_90d is high enough that the engagement pattern is not just noise
On Track / Not Enough Data Yet                               ~0 (monitor only)                         never actioned directly -- re-evaluated next cycle

Impressions_90d distribution within 'Visible & Underclicked' (where triage order should focus):
min       300.0
25%       874.0
50%      2102.0
75%      5708.0
max    517715.0
Name: impressions_90d, dtype: float64


In [8]:
monitoring_triggers = {
    "recompute_cadence": "Every time a fresh data pull is available (this slice is a 90-day trailing window, so re-scoring more than monthly adds little)",
    "decline_rate_drift": f"Recheck if the overall decline rate moves far from the {df['is_declining_label'].mean():.1%} observed here -- a big shift means the label's own baseline has changed",
    "precision_floor": "If a fresh audit's Precision@50 falls below ~0.50 (the low end of Week 6's 5-seed range, 0.52-0.82), stop trusting the ranking until re-validated",
    "new_client_onboarding": "Do not trust this queue for a client until they clear the volume floor (impressions_90d >= 300) for a meaningful share of their pages",
    "queue_concentration_check": "Re-run Week 4's top-50 tier-mix and client-concentration check each cycle; rising concentration is a sign the scoring needs rebalancing, not that one client got worse",
    "model_reassessment": "Re-run the Week 6 multi-seed comparison (not just one split) before ever using the model to do more than provide supporting color",
}
for k, v in monitoring_triggers.items():
    print(f"- {k}: {v}")

- recompute_cadence: Every time a fresh data pull is available (this slice is a 90-day trailing window, so re-scoring more than monthly adds little)
- decline_rate_drift: Recheck if the overall decline rate moves far from the 54.2% observed here -- a big shift means the label's own baseline has changed
- precision_floor: If a fresh audit's Precision@50 falls below ~0.50 (the low end of Week 6's 5-seed range, 0.52-0.82), stop trusting the ranking until re-validated
- new_client_onboarding: Do not trust this queue for a client until they clear the volume floor (impressions_90d >= 300) for a meaningful share of their pages
- queue_concentration_check: Re-run Week 4's top-50 tier-mix and client-concentration check each cycle; rising concentration is a sign the scoring needs rebalancing, not that one client got worse
- model_reassessment: Re-run the Week 6 multi-seed comparison (not just one split) before ever using the model to do more than provide supporting color


In [9]:
export_cols = ["rank", "content_id", "client_id", "archetype", "reason_code", "action_label",
               "opportunity_score", "ctr", "expected_ctr_tier", "ctr_gap", "position_tier",
               "avg_position", "impressions_90d", "sessions_90d", "engagement_rate",
               "content_type", "main_intent", "word_count"]
queue_path = OUT_DIR / "w07_action_playbook_queue.csv"
scored.sort_values("rank")[export_cols].to_csv(queue_path, index=False)
print(f"Wrote {len(scored):,} rows -> {queue_path}")

Wrote 30,000 rows -> work/outputs/w07_action_playbook_queue.csv


In [10]:
fig1, ax1 = plt.subplots(figsize=(6, 4))
action_counts = scored["action_label"].value_counts()
ax1.bar(action_counts.index, action_counts.values, color="#4C72B0")
ax1.set_title("Playbook queue: rows per recommended action")
ax1.set_ylabel("Number of pages")
ax1.tick_params(axis="x", rotation=20)
fig1.tight_layout()
fig1_path = FIG_DIR / "w07_action_counts.png"
fig1.savefig(fig1_path, dpi=150)
plt.close(fig1)
print(f"Saved -> {fig1_path}")

fig2, ax2 = plt.subplots(figsize=(6, 4))
ax2.bar(decay_table.index, decay_table["decline_rate"], color="#DD8452")
ax2.set_title("Decline rate by content-age tier (the decay insight)")
ax2.set_ylabel("Decline rate (%)")
ax2.set_xlabel("age_tier")
fig2.tight_layout()
fig2_path = FIG_DIR / "w07_decline_rate_by_age.png"
fig2.savefig(fig2_path, dpi=150)
plt.close(fig2)
print(f"Saved -> {fig2_path}")

Saved -> work/figures/w07_action_counts.png
Saved -> work/figures/w07_decline_rate_by_age.png


In [11]:
metrics = {
    "seed": SEED,
    "n_rows_scored": int(len(scored)),
    "n_eligible": int(scored["eligible"].sum()),
    "archetype_counts": scored["archetype"].value_counts().to_dict(),
    "action_label_counts": scored["action_label"].value_counts().to_dict(),
    "decay_insight_table": decay_table.reset_index().to_dict(orient="records"),
    "monitoring_triggers": monitoring_triggers,
    "known_limits_summary": [
        "absolute-gap scoring favors page_1 tier",
        "one client can dominate the top of the queue",
        "top_3 tier's expected-CTR benchmark is uncertain (MIXED, Week 4)",
        "model's edge over the rule is modest and inconsistent (3/5 seeds, Week 6)",
        "model's signal mostly overlaps the label's time window (Week 6)",
        "label is a proxy (trend_direction), not a confirmed outcome",
        "only 32 clients -- metrics are noisier than a single number suggests",
    ],
}
metrics_path = OUT_DIR / "w07_playbook_metrics.json"
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2, default=str)
print(f"Wrote receipts -> {metrics_path}")

Wrote receipts -> work/outputs/w07_playbook_metrics.json


In [12]:
checks = {}
checks["ranked_actions_with_reason_codes"] = {"reason_code", "action_label", "opportunity_score"}.issubset(scored.columns)
checks["archetype_action_mapping_present"] = scored["archetype"].nunique() >= 4
checks["decay_insight_present"] = "decay_table" in dir()
checks["intended_use_and_limits_written"] = True
checks["human_review_rules_and_no_go_list_written"] = True
checks["cost_value_thinking_present"] = "cost_value" in dir()
checks["monitoring_retrain_triggers_present"] = "monitoring_triggers" in dir()
checks["queue_csv_exported"] = queue_path.exists()
checks["figures_exported"] = fig1_path.exists() and fig2_path.exists()
checks["metrics_json_exported"] = metrics_path.exists()
checks["not_automated_practical_non_production"] = True

for k, v in checks.items():
    print(f"{'PASS' if v else 'FAIL'} -- {k}")

assert all(checks.values()), "Self-check failed -- see above."
print()
print("Lane confirmed: CTR / Engagement Opportunity Scoring")
print("Reminder: commit work/figures/*.png and work/outputs/*.json -- NOT the queue CSV.")

PASS -- ranked_actions_with_reason_codes
PASS -- archetype_action_mapping_present
PASS -- decay_insight_present
PASS -- intended_use_and_limits_written
PASS -- human_review_rules_and_no_go_list_written
PASS -- cost_value_thinking_present
PASS -- monitoring_retrain_triggers_present
PASS -- queue_csv_exported
PASS -- figures_exported
PASS -- metrics_json_exported
PASS -- not_automated_practical_non_production

Lane confirmed: CTR / Engagement Opportunity Scoring
Reminder: commit work/figures/*.png and work/outputs/*.json -- NOT the queue CSV.
